[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/07_harness/07_build_harness.ipynb)

# 07 · 从零搭建 eval harness

> 《LLM 评测科学》模块 07 配套 notebook。纯 Python / CPU，可选真实小模型（Qwen2.5-0.5B-Instruct）。

讲解（`07_讲解.html`）给出了 harness 的五件套抽象：**Task / Sample / Solver / Scorer / Logger**。
这个 notebook 是"造轮子"模块的主战场：我们从零实现一个约 200 行的 mini harness，把每个设计决策落到代码上。

**构建蓝图**：

1. `Sample` / `Task` 数据结构 + 一个 10 题的 mini 选择题任务
2. `Solver` 协议：确定性、可注入错误的 `mock_solver`（+ 可选的真实模型 `hf_solver`）
3. `Scorer`（`exact_match` / `choice_match`）与 `run_eval()` 最小评测闭环
4. 磁盘缓存层：`key = sha256(model_id + prompt + gen_params)` —— 演示命中、计时、模板变更后自动失效
5. JSONL 日志器：每次调用的完整审计记录
6. 故意注入一个 **harness bug**（抽取器把 `"B."` 解析成 A），演示分数被系统性拉低，再修复对比
7. ✏️ 3 道练习：健壮的 `cache_key`、断点续跑 `resume_run`、数值容差 `numeric_match(tol)`

整个 notebook 不需要网络与 GPU；mock solver 让一切可复现、可断言。

In [ ]:
import hashlib, json, os, pathlib, random, re, shutil, time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone

# ---- 五件套之一/二：Sample 与 Task --------------------------------------
@dataclass(frozen=True)
class Sample:
    id: str          # 稳定 id：断点续跑、跨 run 逐题 diff 都靠它
    question: str
    choices: tuple   # 4 个选项（A/B/C/D 顺序）
    target: str      # 正确选项字母

@dataclass
class Task:
    name: str
    template: str    # prompt 模板属于 Task —— 这是模板可版本化的前提
    samples: list

TEMPLATE_V1 = (
    "Question: {question}\n"
    "Options:\n{options}\n"
    "Answer with a single letter (A/B/C/D)."
)

_QA = [
    ("2 的 10 次方等于多少？", ("512", "1024", "2048", "4096"), "B"),
    ("HTTP 状态码 429 表示什么？", ("服务器内部错误", "资源不存在", "请求过多（限流）", "未授权"), "C"),
    ("SHA-256 的输出长度是多少 bit？", ("128", "256", "512", "1024"), "B"),
    ("Python 中哪种结构保证键的插入顺序（3.7+）？", ("set", "frozenset", "tuple", "dict"), "D"),
    ("JSONL 格式的含义是？", ("JSON 的压缩格式", "嵌套 JSON 数组", "每行一个独立 JSON 对象", "JSON 的二进制编码"), "C"),
    ("temperature=0 的解码近似等价于？", ("贪心解码 greedy", "均匀随机采样", "top-k=40 采样", "beam=4 束搜索"), "A"),
    ("指数退避中第 k 次重试的等待时间通常正比于？", ("k", "2 的 k 次方", "log k", "k 的平方"), "B"),
    ("pass@k 衡量的是什么？", ("k 个模型的平均分", "第 k 题的得分", "k 次重试的延迟", "k 次采样中至少一次通过的概率"), "D"),
    ("few-shot 例子如何选取才可复现？", ("每次随机选", "让模型自己选", "固定 seed 抽样", "人工每次手选"), "C"),
    ("缓存 key 不应该包含以下哪项？", ("model_id", "本次运行的时间戳", "渲染后的 prompt", "生成参数"), "B"),
]

MINI_TASK = Task(
    name="mini_mc_v1",
    template=TEMPLATE_V1,
    samples=[Sample(id=f"q{i:02d}", question=q, choices=c, target=t)
             for i, (q, c, t) in enumerate(_QA)],
)
print(f"Task: {MINI_TASK.name}, {len(MINI_TASK.samples)} samples")
print("gold 分布:", Counter(s.target for s in MINI_TASK.samples))

## Solver 协议：模型调用策略

**Solver** 是 harness 与模型之间的唯一接口，签名固定为：

```python
solve(prompt: str, gen_params: dict) -> str        # 另带 solve.model_id 属性
```

直接生成、CoT、self-consistency、多轮 agent……全部是这个协议的不同实现（模块 06 的引出策略都封装在这一层）。
本 notebook 用两个实现：

- **`mock_solver`**：完全确定性——同一 `(model_id, prompt, gen_params)` 永远返回同一 response（这是缓存演示能用 `assert` 验证的前提）。
  `accuracy` 参数就是**可注入的错误率**：以 `1 - accuracy` 的概率确定性地答错。它还会随机（但确定性地）在三种回答格式间切换：
  `"The answer is (B)."` / `"Answer: B"` / `"B."` —— 第三种格式将在后面触发我们埋的 harness bug。
- **`hf_solver`**（可选）：真实小模型 `Qwen/Qwen2.5-0.5B-Instruct`。需要 `transformers` + `torch`，首次运行下载约 1 GB，CPU 上每题数秒。默认关闭，不影响主线。

In [ ]:
# ---- 五件套之三：Solver --------------------------------------------------
def _stable_seed(*parts):
    # 任意输入 -> 稳定整数种子（跨进程、跨运行一致；不要用内置 hash()，它带随机盐）
    payload = json.dumps(parts, sort_keys=True, ensure_ascii=False)
    return int(hashlib.sha256(payload.encode("utf-8")).hexdigest(), 16) % (2**32)

ANSWER_STYLES = ["The answer is ({}).", "Answer: {}", "{}."]

def make_mock_solver(task, model_id="mock-llm-v1", accuracy=0.8, latency=0.0):
    # 确定性 mock LLM：行为完全由 (model_id, prompt, gen_params) 决定
    gold = {s.question: s.target for s in task.samples}
    def solve(prompt, gen_params):
        if latency:
            time.sleep(latency)            # 模拟网络/推理耗时（缓存计时演示用）
        target = next((t for q, t in gold.items() if q in prompt), "A")
        rng = random.Random(_stable_seed(model_id, prompt, gen_params))
        if rng.random() < accuracy:
            letter = target
        else:
            letter = rng.choice([c for c in "ABCD" if c != target])  # 注入的错误
        return rng.choice(ANSWER_STYLES).format(letter)
    solve.model_id = model_id
    return solve

def make_hf_solver(model_id="Qwen/Qwen2.5-0.5B-Instruct", default_max_new_tokens=32):
    # 可选：真实小模型 Solver。需要 transformers>=4.49 + torch；首次约下载 1GB。
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
    model.eval()
    def solve(prompt, gen_params):
        msgs = [{"role": "user", "content": prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt")
        do_sample = gen_params.get("temperature", 0.0) > 0
        kw = {"max_new_tokens": gen_params.get("max_tokens", default_max_new_tokens),
              "do_sample": do_sample}
        if do_sample:
            kw["temperature"] = gen_params["temperature"]
        with torch.no_grad():
            out = model.generate(**inputs, **kw)
        return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    solve.model_id = model_id
    return solve

USE_HF = False   # 改成 True 即可在后续 cell 用真实模型替换 mock（CPU 可跑，较慢）
print("mock solver 就绪；USE_HF =", USE_HF)

## Scorer 与 `run_eval`：最小评测闭环

**Scorer** 的签名：`scorer(response, target) -> (score, extracted)`。
注意必须把 `extracted`（抽取到的答案）与 `score`（判分）**一起返回**：日志里两者分开记录，事后才能审计"判分的依据"。
抽取失败返回 `extracted=None`，**绝不静默回退到某个默认选项**——这是模块 03 的核心教训，本 notebook 稍后会演示违反它的代价。

`run_eval(task, solver, scorer, seed)` 把五件套串成数据流：

```
Sample --template--> prompt --Solver--> response --Scorer--> (score, extracted) --> record
```

`seed` 进入 `gen_params`，从而进入缓存 key 与日志——任何影响输出的自由度都必须可追溯（讲解 §3 的 config fingerprint 思想）。

In [ ]:
# ---- 五件套之四：Scorer --------------------------------------------------
CHOICE_RE = re.compile(r"\(([A-D])\)|\b([A-D])\b")

def extract_choice(response):
    # 修复版抽取器：覆盖 "(B)" / "Answer: B" / "B." 等格式；失败返回 None
    m = CHOICE_RE.search(response)
    if m is None:
        return None
    return m.group(1) or m.group(2)

def exact_match(response, target):
    ext = response.strip()
    return float(ext == target.strip()), ext

def make_choice_match(extractor):
    def scorer(response, target):
        ext = extractor(response)
        return float(ext == target), ext
    return scorer

choice_match = make_choice_match(extract_choice)

# ---- 评测闭环 ------------------------------------------------------------
def build_prompt(template, sample):
    options = "\n".join(letter + ". " + text
                        for letter, text in zip("ABCD", sample.choices))
    return template.format(question=sample.question, options=options)

def run_eval(task, solver, scorer, seed=0, logger=None):
    # 返回逐题记录（list[dict]）：评测的最小闭环
    gen_params = {"temperature": 0.0, "seed": seed, "max_tokens": 64}
    records = []
    for s in task.samples:
        prompt = build_prompt(task.template, s)
        t0 = time.perf_counter()
        response = solver(prompt, gen_params)
        latency = time.perf_counter() - t0
        score, extracted = scorer(response, s.target)
        rec = {"sample_id": s.id,
               "model_id": getattr(solver, "model_id", "unknown"),
               "prompt": prompt, "response": response,
               "extracted": extracted, "target": s.target, "score": score,
               "latency_s": round(latency, 4),
               "n_prompt_tokens": len(prompt.split()),          # 粗略 token 计数
               "n_completion_tokens": len(response.split()),
               "cached": getattr(solver, "last_hit", False)}
        if logger is not None:
            logger.log(rec)
        records.append(rec)
    return records

def accuracy(records):
    return sum(r["score"] for r in records) / len(records)

solver = make_hf_solver() if USE_HF else make_mock_solver(
    MINI_TASK, model_id="mock-llm-v1", accuracy=0.8, latency=0.03)

records = run_eval(MINI_TASK, solver, choice_match, seed=0)
for r in records:
    flag = "Y" if r["score"] == 1.0 else "x"
    print(f"[{flag}] {r['sample_id']}  response={r['response']!r:30s} extracted={r['extracted']} target={r['target']}")
print(f"\naccuracy = {accuracy(records):.2f}")

## 缓存层与 JSONL 日志器

**缓存**的唯一设计原则：*所有影响 response 的输入必须全部进 key*。

```
key = sha256( model_id ⊕ prompt ⊕ canonicalize(gen_params) )
```

两个工程细节（讲解 §4 的两类事故就源于违反它们）：

1. **规范化**：`gen_params` 用 `json.dumps(..., sort_keys=True)` 序列化，键顺序不影响 key；
2. **边界**：字段间用分隔符（这里用 `\x1f`，ASCII Unit Separator），避免 `("ab","c")` 与 `("a","bc")` 撞 key。

key 基于**渲染后的完整 prompt** 而不是 sample_id —— 模板一变，prompt 即变，key 自动变，旧缓存自然失效，无需手工"清缓存"纪律。

**JSONL 日志**：每次调用追加一行完整记录（prompt / response / extracted / score / 耗时 / token 数 / 是否命中缓存）。
追加写天然抗中断（练习 2 的断点续跑直接读它），原始记录保证"可重放、可争议"。

In [ ]:
# ---- 缓存层 ---------------------------------------------------------------
def cache_key_v1(model_id, prompt, gen_params):
    # v1 实现；练习 1 会让你从零实现并通过更严格的性质测试
    payload = (model_id + "\x1f" + prompt + "\x1f"
               + json.dumps(gen_params, sort_keys=True, ensure_ascii=False))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

class DiskCache:
    def __init__(self, root=".cache"):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.hits = 0
        self.misses = 0
    def get(self, key):
        p = self.root / (key + ".json")
        if p.exists():
            self.hits += 1
            return json.loads(p.read_text(encoding="utf-8"))["response"]
        self.misses += 1
        return None
    def put(self, key, response, meta=None):
        # value 里冗余保存明文 meta（model_id / gen_params）：
        # 哈希是单向的，排查"这条缓存是谁写的"只能靠明文
        payload = {"response": response}
        payload.update(meta or {})
        (self.root / (key + ".json")).write_text(
            json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def with_cache(solver, cache):
    # 把任意 Solver 包装成带缓存的 Solver（对 run_eval 完全透明）
    def solve(prompt, gen_params):
        key = cache_key_v1(solver.model_id, prompt, gen_params)
        hit = cache.get(key)
        if hit is not None:
            solve.last_hit = True
            return hit
        response = solver(prompt, gen_params)
        cache.put(key, response,
                  meta={"model_id": solver.model_id, "gen_params": gen_params})
        solve.last_hit = False
        return response
    solve.model_id = solver.model_id
    solve.last_hit = False
    return solve

# ---- 五件套之五：Logger ----------------------------------------------------
class JSONLLogger:
    def __init__(self, path):
        self.path = pathlib.Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
    def log(self, record):
        rec = dict(record)
        rec["ts"] = datetime.now(timezone.utc).isoformat()
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    def read(self):
        if not self.path.exists():
            return []
        with open(self.path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]

print("DiskCache / JSONLLogger / with_cache 就绪")

In [ ]:
# 演示：冷缓存 -> 热缓存 -> 模板变更后自动失效
shutil.rmtree(".cache", ignore_errors=True)        # 清空，保证从冷缓存开始
if os.path.exists("logs/run_demo.jsonl"):
    os.remove("logs/run_demo.jsonl")

CACHE = DiskCache(".cache")
LOG = JSONLLogger("logs/run_demo.jsonl")
cached_solver = with_cache(solver, CACHE)

t0 = time.perf_counter()
rec_cold = run_eval(MINI_TASK, cached_solver, choice_match, seed=0, logger=LOG)
t_cold = time.perf_counter() - t0

t0 = time.perf_counter()
rec_warm = run_eval(MINI_TASK, cached_solver, choice_match, seed=0, logger=LOG)
t_warm = time.perf_counter() - t0

print(f"冷缓存: {t_cold:.3f}s | 热缓存: {t_warm:.3f}s "
      f"(约 {t_cold / max(t_warm, 1e-9):.0f}x 加速) | hits={CACHE.hits} misses={CACHE.misses}")
assert CACHE.hits == 10 and CACHE.misses == 10
assert [r["response"] for r in rec_cold] == [r["response"] for r in rec_warm], "确定性被破坏！"
assert not any(r["cached"] for r in rec_cold) and all(r["cached"] for r in rec_warm)

# 模板变更 -> 渲染后的 prompt 变 -> key 变 -> 缓存自动失效（无需手工清理）
task_v2 = Task(name=MINI_TASK.name + "-template-v2",
               template=TEMPLATE_V1 + "\nLet's think step by step.",
               samples=MINI_TASK.samples)
misses_before = CACHE.misses
rec_v2 = run_eval(task_v2, cached_solver, choice_match, seed=0, logger=LOG)
print(f"模板 v2 之后: misses {misses_before} -> {CACHE.misses}（10 题全部未命中，旧缓存自动失效）")
assert CACHE.misses == misses_before + 10

print("JSONL 日志行数:", len(LOG.read()), "（3 个 run x 10 题，每次调用一行，含命中缓存的调用）")

## 注入一个 harness bug：抽取器把 `"B."` 解析成 A

把模块 03 的教训工程化。下面是一个**真实事故模式的简化复刻**：抽取正则只认 `"answer is (X)"` 一种格式，
没匹配到就走 `return "A"` 的默认分支——**静默 fallback 是评测工程的头号反模式**。

后果有两重：

1. 分数被**系统性**拉低：所有用 `"Answer: B"` / `"B."` 格式回答的题同时受害——这不是随 $n$ 缩小的统计噪声，置信区间罩不住它；
2. 抽取分布向 A 倾斜：你可能把它误诊为模型的"选项位置偏差"（模块 03），在错误的方向上排查很久。

正确做法：抽取失败返回 `None` 并计数上报；**抽取失败率本身是必须监控的指标**。

In [ ]:
def buggy_extract(response):
    # BUG 版抽取器：只认 "answer is (X)"；匹配失败时静默回退到 "A"
    m = re.search(r"answer is \(([A-D])\)", response, re.IGNORECASE)
    return m.group(1) if m else "A"

probes = ["The answer is (B).", "Answer: B", "B."]
print("response               -> buggy | fixed")
for p in probes:
    print(f"{p!r:22s} ->  {buggy_extract(p)}    |  {extract_choice(p)}")

buggy_scorer = make_choice_match(buggy_extract)
rec_buggy = run_eval(MINI_TASK, solver, buggy_scorer, seed=0)
rec_fixed = run_eval(MINI_TASK, solver, choice_match, seed=0)
acc_buggy, acc_fixed = accuracy(rec_buggy), accuracy(rec_fixed)

print(f"\nbuggy extractor: accuracy = {acc_buggy:.2f}")
print(f"fixed extractor: accuracy = {acc_fixed:.2f}   <- 同一批 response，仅判分器不同")
for b, f in zip(rec_buggy, rec_fixed):
    if b["extracted"] != f["extracted"]:
        print(f"  {b['sample_id']}: response={b['response']!r}  buggy 抽成 {b['extracted']} / fixed 抽成 {f['extracted']}")
assert acc_buggy <= acc_fixed

# 第二重危害：抽取分布向默认值 A 倾斜
print("buggy 抽取分布:", dict(Counter(r["extracted"] for r in rec_buggy)))
print("fixed 抽取分布:", dict(Counter(r["extracted"] for r in rec_fixed)))
# 顺带演示"可重放"：判分器修复后无需重新调用模型——
# 直接用日志/记录里的 response 重新过一遍 Scorer 即可（生成与判分解耦的红利）
replayed = [make_choice_match(extract_choice)(r["response"], r["target"])[0] for r in rec_buggy]
assert sum(replayed) / len(replayed) == acc_fixed
print("离线重放（不调模型）修正后的 accuracy =", f"{sum(replayed) / len(replayed):.2f}")

In [ ]:
# 我们造的轮子 vs 生产级框架 Inspect (UK AISI 2024) 的等价写法
# 安装：pip install inspect-ai   （未安装时本 cell 只打印对照说明，不会报错）
try:
    from inspect_ai import Task as InspectTask, task
    from inspect_ai.dataset import Sample as InspectSample
    from inspect_ai.solver import multiple_choice
    from inspect_ai.scorer import choice

    @task
    def mini_mc():
        dataset = [InspectSample(input=s.question, choices=list(s.choices), target=s.target)
                   for s in MINI_TASK.samples]
        return InspectTask(dataset=dataset, solver=multiple_choice(), scorer=choice())

    print("inspect_ai 已安装。把本任务存成 .py 后即可：")
    print("  inspect eval mini_mc.py --model hf/Qwen/Qwen2.5-0.5B-Instruct")
    print("  inspect view    # 打开日志查看器，逐题审计 prompt/response/score")
except ImportError:
    print("未安装 inspect_ai。我们的 mini harness 与 Inspect 的对应关系：")
    print("  Sample / Task        -> inspect_ai.dataset.Sample / inspect_ai.Task")
    print("  mock/hf solver       -> inspect_ai.solver（generate、multiple_choice，可组合成链）")
    print("  Scorer               -> inspect_ai.scorer（choice、match、model_graded_qa）")
    print("  JSONLLogger          -> .eval 日志 + `inspect view` 图形化审计")
    print("  DiskCache            -> 内置调用级缓存（generate(..., cache=True)）")

## ✏️ 练习 1：实现健壮的 `cache_key`

实现 `cache_key(model_id, prompt, params) -> str`（64 位十六进制 SHA-256），满足三条性质：

1. **params 顺序无关**：键顺序（含嵌套 dict）不同但内容相同 → key 相同；
2. **任何字段变化 key 必变**：model_id / prompt / params 任一处不同 → key 不同；
3. **边界无歧义**：`("ab", "c")` 与 `("a", "bc")` 不得撞 key（直接字符串拼接会在这里翻车）。

**提示**：把三个字段放进一个 `list` 再 `json.dumps(..., sort_keys=True)` 一次性序列化，三条性质同时满足。10 行以内可完成。

In [ ]:
def cache_key(model_id: str, prompt: str, params: dict) -> str:
    # TODO:
    #  1) 把 (model_id, prompt, params) 序列化为一个无歧义、规范化的字符串
    #  2) 返回其 SHA-256 十六进制摘要（64 个字符）
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----------------------------------------------------------
k_base = cache_key("m1", "hello", {"temperature": 0.0, "seed": 1})
assert isinstance(k_base, str) and len(k_base) == 64
assert all(c in "0123456789abcdef" for c in k_base)
# 性质 1：参数顺序无关（含嵌套 dict）
assert cache_key("m1", "hello", {"seed": 1, "temperature": 0.0}) == k_base
assert cache_key("m", "p", {"opts": {"x": 1, "y": 2}}) == cache_key("m", "p", {"opts": {"y": 2, "x": 1}})
# 性质 2：任何字段变化 key 必变
assert cache_key("m2", "hello", {"temperature": 0.0, "seed": 1}) != k_base
assert cache_key("m1", "hello!", {"temperature": 0.0, "seed": 1}) != k_base
assert cache_key("m1", "hello", {"temperature": 0.7, "seed": 1}) != k_base
assert cache_key("m1", "hello", {"temperature": 0.0, "seed": 1, "stop": ["\n"]}) != k_base
# 性质 3：边界无歧义
assert cache_key("ab", "c", {}) != cache_key("a", "bc", {})
print("✅ 练习 1 通过")

## ✏️ 练习 2：断点续跑 `resume_run`

长评测中途被打断（机器重启、限流封禁）是常态。利用 JSONL 日志的追加写特性实现断点续跑：

实现 `resume_run(task, solver, scorer, log_path, seed=0) -> list[dict]`：

1. 用 `JSONLLogger(log_path).read()` 读出已完成记录，按 `sample_id` 建立"已完成"映射；
2. 遍历 `task.samples`：已完成的**直接复用旧记录、不调 solver**；未完成的正常评测并写日志；
3. 按 `task.samples` 顺序返回完整 records。

**提示**：复用 `build_prompt`，并用与 `run_eval` 完全一致的 `gen_params` 构造（`{"temperature": 0.0, "seed": seed, "max_tokens": 64}`）。15 行左右。

In [ ]:
def resume_run(task, solver, scorer, log_path, seed=0):
    # TODO:
    #  1) done = {r["sample_id"]: r for r in JSONLLogger(log_path).read()}
    #  2) 对每个 sample：已在 done 里 -> 复用；否则评测 + logger.log(rec)
    #  3) 按 task.samples 顺序返回全部 records
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----------------------------------------------------------
ex2_log = "logs/exercise2_resume.jsonl"
if os.path.exists(ex2_log):
    os.remove(ex2_log)

calls = {"n": 0}
_base_solver = make_mock_solver(MINI_TASK, model_id="resume-test-model", accuracy=0.8)
def counting_solver(prompt, gen_params):
    calls["n"] += 1
    return _base_solver(prompt, gen_params)
counting_solver.model_id = "resume-test-model"

# 模拟"跑到第 4 题被中断"：只评前 4 题并写日志
partial_task = Task(MINI_TASK.name, MINI_TASK.template, MINI_TASK.samples[:4])
run_eval(partial_task, counting_solver, choice_match, seed=0, logger=JSONLLogger(ex2_log))
assert calls["n"] == 4

# 断点续跑：只应再调用 6 次
records_resumed = resume_run(MINI_TASK, counting_solver, choice_match, ex2_log, seed=0)
assert calls["n"] == 10, f"应该总共调用 10 次（4 + 6），实际 {calls['n']}"
assert len(records_resumed) == 10
assert [r["sample_id"] for r in records_resumed] == [s.id for s in MINI_TASK.samples]
# 再续跑一次：全部已完成，0 次新调用
resume_run(MINI_TASK, counting_solver, choice_match, ex2_log, seed=0)
assert calls["n"] == 10, "全部已完成时不应有任何新调用"
print("✅ 练习 2 通过")

## ✏️ 练习 3：新增数值容差 Scorer `numeric_match(tol)`

数学/物理类任务的答案是数值，逐字符 `exact_match` 会把 `3.1416` 与 `3.14159` 判错。
实现 scorer 工厂 `numeric_match(tol)`，返回的 scorer 满足本 harness 的统一协议 `scorer(response, target) -> (score, extracted)`：

1. 用给定的 `NUM_RE` 抽取 response 中所有数字，取**最后一个**作为模型答案（约定：结论写在最后）；
2. 抽不到数字 → `(0.0, None)`；
3. `|answer - float(target)| <= tol` → `(1.0, answer)`，否则 `(0.0, answer)`。

**提示**：`NUM_RE.findall` + `float()`，10 行以内。写完它就能直接插进 `run_eval`——这就是 Scorer 协议正交性的意义。

In [ ]:
NUM_RE = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")

def numeric_match(tol):
    # TODO: 返回 scorer(response, target) -> (score, extracted)
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----------------------------------------------------------
nm = numeric_match(0.01)
s, x = nm("The result is 3.14159", "3.14")
assert s == 1.0 and abs(x - 3.14159) < 1e-9
s, _ = numeric_match(0.0001)("The result is 3.14159", "3.14")
assert s == 0.0                       # 容差收紧后判错
s, x = nm("no number here", "3.14")
assert s == 0.0 and x is None         # 抽取失败 -> None，绝不默认猜
s, _ = numeric_match(0.5)("answer: -2.7", "-3")
assert s == 1.0                       # 负数 + 容差
s, _ = nm("先得到 1.0，最终答案是 2.0", "2.0")
assert s == 1.0                       # 取最后一个数字
s, _ = numeric_match(0)("42", "42")
assert s == 1.0                       # 整数、零容差
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。三题的参考实现如下。

In [ ]:
# 练习 1 参考答案 —— 先自己做，再对照
def cache_key(model_id: str, prompt: str, params: dict) -> str:
    # 一次性序列化整个 list：
    #  - list 结构给出无歧义的字段边界（性质 3）
    #  - sort_keys=True 递归规范化所有 dict 键序（性质 1）
    #  - 任一字段变化 -> payload 变 -> SHA-256 摘要变（性质 2）
    payload = json.dumps([model_id, prompt, params],
                         sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

print("练习 1 参考实现已加载，可回到自测 cell 重新运行")

In [ ]:
# 练习 2 参考答案 —— 先自己做，再对照
def resume_run(task, solver, scorer, log_path, seed=0):
    logger = JSONLLogger(log_path)
    done = {r["sample_id"]: r for r in logger.read()}   # 已完成 -> 跳过
    gen_params = {"temperature": 0.0, "seed": seed, "max_tokens": 64}
    records = []
    for s in task.samples:
        if s.id in done:
            records.append(done[s.id])
            continue
        prompt = build_prompt(task.template, s)
        t0 = time.perf_counter()
        response = solver(prompt, gen_params)
        latency = time.perf_counter() - t0
        score, extracted = scorer(response, s.target)
        rec = {"sample_id": s.id, "model_id": getattr(solver, "model_id", "unknown"),
               "prompt": prompt, "response": response,
               "extracted": extracted, "target": s.target, "score": score,
               "latency_s": round(latency, 4)}
        logger.log(rec)          # 边跑边落盘：下次中断仍可续
        records.append(rec)
    return records

print("练习 2 参考实现已加载，可回到自测 cell 重新运行")

In [ ]:
# 练习 3 参考答案 —— 先自己做，再对照
def numeric_match(tol):
    def scorer(response, target):
        nums = NUM_RE.findall(response)
        if not nums:
            return 0.0, None     # 抽取失败 -> None，绝不默认猜（与选择题抽取器同一纪律）
        answer = float(nums[-1])
        return float(abs(answer - float(target)) <= tol), answer
    return scorer

print("练习 3 参考实现已加载，可回到自测 cell 重新运行")

## 小结

这个不到 200 行的 mini harness 已经具备了生产级 harness 的全部"骨架"：

- **五件套抽象**（Task / Sample / Solver / Scorer / Logger）正交解耦：换 Solver 不动 Scorer，加 Scorer（练习 3）不碰其它任何代码；
- **缓存**：key = `sha256(model_id ⊕ prompt ⊕ canonical(gen_params))`，基于渲染后的 prompt——模板变更自动失效，参数全量进 key 杜绝脏命中；
- **JSONL 日志**：每次调用完整落盘，支撑断点续跑（练习 2）、离线重放（判分器修复后不重调模型）、逐题争议仲裁；
- **harness bug 的本质**：不报错、不崩溃，只是安静地给出错误的分数——所以 `extracted` 必须入日志、抽取失败必须返回 `None` 并计数、判分器改动必须触发历史重放。

真正的生产系统在此之上还要加：异步并发 + 限流退避、失败题的统计口径（记 0 / 记 NaN / 重跑，见讲解 §6 的偏差公式）、config fingerprint 与版本钉死。
能自己写出这套骨架之后，再去读 Inspect [UK AISI 2024] 与 lm-evaluation-harness [Gao 2023] 的源码，你看到的就不再是 API，而是一连串你已经亲手踩过的设计决策；
[Biderman 2024]（*Lessons from the Trenches on Reproducible Evaluation*）则是这些决策的事故复盘合集，强烈建议通读。

**下一模块（08 · Arena、time-horizon 与评测报告）**：harness 产出的是逐题记录，最终要变成可发布的结论——Elo / Bradley–Terry 配对比较、METR 的 time-horizon 度量，以及一份合格评测报告必须包含什么。

---
## 🎯 真实数据胶囊题：在真实 GSM8K 上跑一个完整的 exact-match harness

把 eval 的全链路串起来：取真实题目 -> 解析金标 -> 抽取预测 -> 精确匹配打分 -> 聚合 + bootstrap CI。这就是一个最小但完整的评测 harness。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.llm_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(ans): return ans.split("####")[-1].strip().replace(",","")
def shakespeare():
    return open(_f("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

rows=gsm8k(150)
def parse_pred(text):
    nums=re.findall(r"-?\d+\.?\d*", text.replace(",",""))
    return nums[-1].rstrip("0").rstrip(".") if nums else None
# 模拟模型输出：65% 给对，35% 给一个错数
rng=np.random.default_rng(0)
def model_output(g):
    return f"... so the answer is {g}." if rng.random()<0.65 else f"answer: {rng.integers(0,999)}"
print("harness 输入: 真实 GSM8K", len(rows), "题")

**练习**：实现 `run_harness(rows)`：对每题用 `model_output` 生成、`parse_pred` 抽取、与 `gold()` 精确匹配，返回 `(accuracy, correct_array)`。

In [ ]:
def run_harness(rows):
    # TODO: 遍历 rows，预测 vs gold 精确匹配，返回 (acc, 0/1数组)
    raise NotImplementedError


In [ ]:
# 自测
acc, correct = run_harness(rows)
assert len(correct)==len(rows) and set(np.unique(correct)).issubset({0.0,1.0})
assert 0.5 < acc < 0.8, "约 65% 正确率"
# 配 bootstrap CI
rng2=np.random.default_rng(1)
boots=[correct[rng2.integers(0,len(correct),len(correct))].mean() for _ in range(1000)]
lo,hi=np.percentile(boots,[2.5,97.5])
print(f"harness 结果: acc={acc:.3f}  95% CI=[{lo:.3f},{hi:.3f}] ✓")


### 📖 参考答案

In [ ]:
def run_harness(rows):
    correct=[]
    for r in rows:
        pred=parse_pred(model_output(gold(r["answer"])))
        correct.append(1.0 if pred==gold(r["answer"]) else 0.0)
    correct=np.array(correct); return float(correct.mean()), correct
print("✓ 一个 harness = 取数据+生成+抽取+打分+聚合+误差棒，缺一不可")